# Information map: the first filled cells


**What this writes (instruction 72).** Filled cells, only from questions whose frozen test cleared the 90 % precision bar at its lower bound, only where the calibrated probability of yes reaches the threshold that produced that precision. Everything else the methods say stays a ranking, outside the map.

Left out although they may fill, because the label is not biology: fit_density_dependence_log10padj, crispr_gra17_candidate, n_holes.

Built 2026-11-08.

In [1]:
import json, os, numpy as np, pandas as pd
from parascape import datasets as ds, harness as H, harness_answer as HA, information_categories as IC, organisms as O
from parascape.question_registry import Question
S = pd.read_csv(os.path.join(REPO, 'results/question_scoreboard_2026_11_08/scores.tsv'), sep='\t')
chosen = S[S.may_fill & ~S.column.isin(NOT_BIOLOGY)]
nodes = pd.read_parquet(os.path.join(REPO, 'parascape/data/nodes.parquet')).set_index('gene_id')
numeric = [c for c in nodes.columns if pd.api.types.is_numeric_dtype(nodes[c]) or pd.api.types.is_bool_dtype(nodes[c])]
fam = lambda c: (ds.provenance(c, O.TOXOPLASMA).key if ds.provenance(c, O.TOXOPLASMA) else 'unregistered')
families = pd.Series({c: fam(c) for c in numeric})
groups = nodes['orthogroup'].astype(object)
groups = groups.where(groups.notna(), pd.Series({g: 'self:' + g for g in nodes.index}))
cat = pd.read_csv(os.path.join(REPO, 'results/question_catalogue_2026_11_08/catalogue.tsv'), sep='\t').set_index('column')
chosen[['question', 'column', 'category', 'frozen_auc', 'precision_lower90']]

,question,column,category,frozen_auc,precision_lower90
92,cat_invivo_WholeBrain_Acute_1_change,invivo_WholeBrain_Acute_1,transcript,0.8743,0.9157
105,cat_lopit_unified_nucleus,lopit_unified,place,0.9141,0.9259
108,cat_macrophage_expression_percentile_high,macrophage_expression_percentile,transcript,0.9687,0.9012
161,cat_rpf129869_confluent_r1_change,rpf129869_confluent_r1,translation,0.9854,0.9042


## 1 · Re-run, deploy, keep the fills

In [2]:
fills, summary = [], []
TAG = 'filled:hist_gbt@2026-11-08'
for r in chosen.itertuples(index=False):
    row = cat.loc[r.column]
    for suffix, wording, labels in HA.binary_labels(r.column, row.kind, row.category, nodes[r.column]):
        qid = f'cat_{HA._safe(r.column)}_{suffix}'
        if qid != r.question:
            continue
        family = fam(r.column)
        q = Question(id=qid, text=f'{row.question} -- yes when {wording}', answer_type='boolean', question_class='B',
                     label_source=f'nodes.parquet:{r.column}', label_family=family, evidence_forbidden=(family,),
                     label_tier='observed-screen', label_category=row.category, status='scored')
        full = nodes.loc[labels.index, numeric].astype('float64')
        feats = H.allowed_features(q, full, families)
        res = H.run_question(q, labels=labels.astype(bool), features=feats, families=families,
                             groups=groups.reindex(labels.index),
                             strategies=[H.Strategy.of('boosted trees', 'hist_gbt', max_iter=150)], seed=0,
                             importance_repeats=1, log_path=H.HOLDOUT_LOG)
        unl = nodes.index[nodes[r.column].isna()]
        if not len(unl):   # every gene is measured: nothing to fill
            summary.append(dict(column='filled_' + qid[len('cat_'):], question=q.text, unlabelled=0, called=0, filled=0,
                                may_fill=res.may_fill, frozen_lower90=round(res.frozen_test['precision_lower90'], 3), flags=', '.join(res.flags)))
            continue
        dep = H.deploy(res, nodes.loc[unl, feats.columns].astype('float64'))
        hit = dep[dep.fills]
        # Applicability: the 90 % precision was measured on labelled genes. If the genes without a label are
        # filled far more often than the label occurs (> 3x its prevalence), they are a different population
        # (missing not at random) and the measured precision does not carry over: no fills, reason recorded.
        prevalence = float(labels.mean()); rate = len(hit) / len(unl)
        applicable = rate <= 3 * prevalence
        if not applicable:
            hit = hit.iloc[:0]
        col = 'filled_' + qid[len('cat_'):]
        for g, p in hit.calibrated.items():
            fills.append(dict(gene_id=g, trait=col, value='True', probability=round(float(p), 4),
                              source_tier=f'{TAG} {qid}', reference='results/question_scoreboard_2026_11_08',
                              dataset='question catalogue (instruction 73)'))
        summary.append(dict(column=col, question=q.text, unlabelled=len(unl), called=int(dep.called.sum()),
                            filled=len(hit), fill_rate=round(rate, 3), prevalence=round(prevalence, 3), applicable=applicable,
                            may_fill=res.may_fill, frozen_lower90=round(res.frozen_test['precision_lower90'], 3),
                            flags=', '.join(res.flags)))
F = pd.DataFrame(fills); Sm = pd.DataFrame(summary)
F.to_csv(os.path.join(OUT, 'fills.tsv'), sep='\t', index=False); Sm.to_csv(os.path.join(OUT, 'summary.tsv'), sep='\t', index=False)
Sm

,column,question,unlabelled,called,filled,fill_rate,prevalence,applicable,may_fill,frozen_lower90,flags
0,filled_invivo_WholeBrain_Acute_1_change,Does the gene respond in invivo WholeBrain Acute 1? -- yes when among the largest 20 % of changes (|invivo_WholeBrain_Acute_1| >= 5.04),477,19,19,0.04,0.394,True,True,0.916,frozen_test_repeat
1,filled_lopit_unified_nucleus,Where is the protein (lopit unified)? -- yes when lopit_unified = nucleus,4313,9,9,0.002,0.33,True,True,0.926,frozen_test_repeat
2,filled_macrophage_expression_percentile_high,Does the gene respond in macrophage expression percentile? -- yes when among the highest 20 % (macrophage_expression_percentile >= 80.5),0,0,0,NaN,NaN,NaN,True,0.901,frozen_test_repeat
3,filled_rpf129869_confluent_r1_change,Is the gene translated differently in rpf129869 confluent r1? -- yes when among the largest 20 % of changes (|rpf129869_confluent_r1| >= 4.32),260,254,0,0.977,0.235,False,True,0.904,frozen_test_repeat


## 2 · Into the published map

One column per question, holding only its fills; probability = calibrated probability; source = the tag and the question. Evidence rows appended in the same form. Observed columns are not touched.

In [3]:
values = pd.read_parquet(os.path.join(MAP, 'values.parquet')); prob = pd.read_parquet(os.path.join(MAP, 'probability.parquet'))
source = pd.read_parquet(os.path.join(MAP, 'source.parquet')); evidence = pd.read_parquet(os.path.join(MAP, 'evidence.parquet'))
before = {k: v.shape for k, v in dict(values=values, probability=prob, source=source, evidence=evidence).items()}
for col in Sm[Sm.filled > 0].column:
    for frame in (values, prob, source):
        if col in frame.columns:
            frame.drop(columns=col, inplace=True)
    rows = F[F.trait == col].set_index('gene_id')
    values[col] = rows.value.map(lambda v: v == 'True').reindex(values.index).astype('object')
    prob[col] = rows.probability.reindex(values.index)
    source[col] = rows.source_tier.reindex(values.index)
evidence = evidence[~evidence.trait.isin(Sm.column)]
evidence = pd.concat([evidence, F[evidence.columns.intersection(F.columns)]], ignore_index=True)
values.to_parquet(os.path.join(MAP, 'values.parquet')); prob.to_parquet(os.path.join(MAP, 'probability.parquet'))
source.to_parquet(os.path.join(MAP, 'source.parquet')); evidence.to_parquet(os.path.join(MAP, 'evidence.parquet'), index=False)
desc_path = os.path.join(MAP, 'column_descriptions.json')
desc = json.load(open(desc_path)) if os.path.exists(desc_path) else {}
desc.update({r.column: f'FILLED (predicted, never observed): {r.question}' for r in Sm[Sm.filled > 0].itertuples(index=False)})
json.dump(desc, open(desc_path, 'w'), indent=1)
after = {k: v.shape for k, v in dict(values=values, probability=prob, source=source, evidence=evidence).items()}
from parascape import information_map as IM
m = IM.load_published(MAP)
print(before, '->', after)
print(m.banner())
m.predicted_cells().groupby('column').size().to_frame('filled cells')

{'values': (8140, 20), 'probability': (8140, 20), 'source': (8140, 20), 'evidence': (3726, 7)} -> {'values': (8140, 22), 'probability': (8140, 22), 'source': (8140, 22), 'evidence': (3754, 7)}
28 cells of this release are predicted (filled or exploratory). Check their source tier before reading one as a fact.


,filled cells
column,
filled_invivo_WholeBrain_Acute_1_change,19
filled_lopit_unified_nucleus,9
